# 04 Signal Generation

Train the primary XGBoost model with walk-forward retraining and compare it with benchmark classifiers.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

import pandas as pd
from data_loader import load_ohlcv_csv
from signal_model import (
    create_forward_return_targets,
    evaluate_classification_predictions,
    walk_forward_train_predict,
)
from utils import project_path

In [ ]:
prices = load_ohlcv_csv(project_path("data", "raw", "SPY.csv"))
features = pd.read_csv(project_path("data", "processed", "SPY_features.csv"), index_col=0, parse_dates=True)
regimes = pd.read_csv(project_path("data", "processed", "SPY_regimes.csv"), index_col=0, parse_dates=True)

model_input = features.join(regimes)
targets = create_forward_return_targets(
    prices,
    horizon=5,
    positive_threshold=0.02,
    label_mode="binary",
)

model_metrics = {}
for model_name in ["xgboost", "logistic_regression", "random_forest"]:
    predictions = walk_forward_train_predict(
        model_input,
        targets["target"],
        model_name=model_name,
        min_train_size=252 * 2,
        test_window=21,
        step_size=21,
        random_state=42,
    )
    predictions.to_csv(project_path("data", "processed", f"SPY_predictions_{model_name}.csv"))
    model_metrics[model_name] = evaluate_classification_predictions(
        predictions["target"],
        predictions["prediction"],
        predictions.filter(like="class_"),
    )

pd.DataFrame(model_metrics).T.sort_values("roc_auc", ascending=False)